# LLMs for Text Transformation

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [12]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [19]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [22]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [24]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [26]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: ```Je veux commander un ballon de basket```

Spanish: ```Quiero ordenar un balón de baloncesto```

Pirate English: ```I be wantin' to order a basketball```


In [27]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [31]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [33]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."
Korean: "내 모니터에는 불이 켜지지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese. It translates to "My screen is flickering" in English.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [36]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Dear Sir/Madam,

I am writing to bring to your attention the specifications of a standing lamp that I believe may be of interest to you. 

Sincerely,
Joe


## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [39]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
<title>Restaurant Employees</title>
</head>
<body>

<table border="1">
  <tr>
    <th>Name</th>
    <th>Email</th>
  </tr>
  <tr>
    <td>Shyam</td>
    <td>shyamjaiswal@gmail.com</td>
  </tr>
  <tr>
    <td>Bob</td>
    <td>bob32@gmail.com</td>
  </tr>
  <tr>
    <td>Jai</td>
    <td>jai87@gmail.com</td>
  </tr>
</table>

</body>
</html>


In [41]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [44]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball.
No errors found
No errors found.
Their goes my freedom. There going to bring they’re suitcases.

No errors found.

Rewritten: Their goes my freedom. There going to bring their suitcases.
You're going to need your notebook.
That medicine affects my ability to sleep. Have you heard of the butterfly effect?
No errors found


In [45]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.


In [53]:
# ! pip3 install redlines

In [55]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

<span style='color:red;font-weight:700;text-decoration:line-through;'>Got </span><span style='color:green;font-weight:700;'>I got </span>this for my daughter for her birthday <span style='color:red;font-weight:700;text-decoration:line-through;'>cuz </span><span style='color:green;font-weight:700;'>because </span>she keeps taking mine from my <span style='color:red;font-weight:700;text-decoration:line-through;'>room.  </span><span style='color:green;font-weight:700;'>room. </span>Yes, adults also like pandas <span style='color:red;font-weight:700;text-decoration:line-through;'>too.  </span><span style='color:green;font-weight:700;'>too. </span>She takes it everywhere with her, and it's super soft and <span style='color:red;font-weight:700;text-decoration:line-through;'>cute.  </span><span style='color:green;font-weight:700;'>cute. </span>One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same <span style='color:red;font-weight:700;text-decoration:line-through;'>price.  </span><span style='color:green;font-weight:700;'>price. </span>It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.

In [57]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

I purchased this adorable panda plush as a birthday gift for my daughter, as she kept borrowing mine from my room. It's not just for kids - even adults can appreciate the charm of pandas. The plush is incredibly soft and cute, and my daughter carries it everywhere with her. However, I did notice that one of the ears is slightly lower than the other, which seems like a manufacturing flaw rather than intentional design. While I found the plush to be a bit smaller than expected given the price, I believe there are larger options available for the same cost. Despite this, the plush arrived a day earlier than anticipated, allowing me to enjoy it myself before passing it on to my daughter. Overall, it's a delightful gift that brings joy to both children and adults alike.

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

---

## Ma partie : trois transformations, chacune verifiee automatiquement

Les trois versions portent sur des comptes rendus d'EEG **fictifs** (aucune donnee patient reelle),
un texte que je manipule tous les jours. L'interet n'est pas seulement de varier les prompts : dans
les exemples du cours, on juge la sortie a l'oeil. Ici chaque version est accompagnee d'un
**controle automatique en Python**, parce qu'a l'oeil on ne voit pas qu'un chiffre a bouge.

| Version | Transformation | Ce qui est mesure automatiquement |
|---|---|---|
| V1 | traduction FR -> EN d'un compte rendu technique | tous les nombres, unites et acronymes du texte source sont-ils intacts dans la traduction ? |
| V2 | compte rendu technique -> explication grand public | le modele ajoute-t-il de l'information absente du source (diagnostic, traitement, pronostic) ? |
| V3 | texte libre -> JSON structure | le JSON est-il valide, et le modele met-il `null` quand l'information manque, ou invente-t-il une valeur ? |

Le fil conducteur : **une transformation de texte reussie est une transformation qui ne cree rien.**
C'est exactement ce que les trois controles cherchent a prendre en defaut.

In [ ]:
# --- Corpus de test : comptes rendus EEG fictifs -------------------------------
# Textes inventes pour l'exercice, aucune donnee patient.

comptes_rendus = [
    """Enregistrement EEG de veille et de sommeil de 30 minutes, 21 electrodes,
    montage 10-20, echantillonnage a 256 Hz. Rythme de fond alpha a 9,5 Hz,
    reactif a l'ouverture des yeux, symetrique. Presence de pointes-ondes
    temporales anterieures gauches, isolees, d'amplitude maximale 120 microvolts,
    activees par l'endormissement. Pas d'anomalie a l'hyperpnee de 3 minutes ni
    a la stimulation lumineuse intermittente jusqu'a 20 Hz.""",

    """Monitoring video-EEG de longue duree, 5 jours, 64 canaux. Trois evenements
    cliniques enregistres a J2 et J3. Duree moyenne 45 secondes. Debut par une
    sensation epigastrique ascendante, suivie d'une rupture de contact et
    d'automatismes oro-alimentaires. Correlat EEG : decharge rythmique theta a
    5 Hz debutant sur les electrodes temporales droites.""",

    """EEG de 20 minutes chez un adulte. Trace normal. Rythme de fond a 10 Hz,
    bien organise, reactif. Aucune anomalie paroxystique. Hyperpnee et
    stimulation lumineuse sans particularite.""",
]

print(len(comptes_rendus), "comptes rendus de test")
print(comptes_rendus[0][:120], "...")

### V1 — Traduction technique : est-ce que les chiffres survivent ?

Traduire un compte rendu FR -> EN parait anodin. Le risque reel n'est pas la grammaire : c'est
qu'un `9,5 Hz` devienne `9.5 Hz` (correct) ou `95 Hz` (faux), qu'une unite disparaisse, ou qu'un
acronyme normalise soit "traduit". Sur un compte rendu, une valeur alteree change le sens medical
sans que rien dans le texte ne signale l'erreur.

Le prompt impose donc trois contraintes explicites, et le controle Python verifie qu'elles ont
ete tenues.

In [ ]:
import re

ACRONYMES_PROTEGES = ["EEG", "Hz", "10-20", "video-EEG"]


def extraire_nombres(texte):
    """Nombres avec leur unite eventuelle, virgule decimale normalisee en point."""
    trouves = re.findall(r"\d+(?:[.,]\d+)?", texte)
    return sorted({t.replace(",", ".") for t in trouves})


def traduire_v1(texte):
    prompt = f"""Translate the following French clinical EEG report into English.

    Strict constraints:
    - Keep every number exactly as it is. Convert the French decimal comma to a decimal point,
      but never change the value itself.
    - Keep every unit (Hz, microvolts, minutes, seconds, days, channels) and never convert
      between units.
    - Do not translate standard technical acronyms: EEG, video-EEG, Hz, the 10-20 montage.
    - Do not add, interpret or summarise anything. Translate only what is written.
    - Output the translation only, with no preamble.

    Report: ```{texte}```"""
    return get_completion(prompt)


resultats_v1 = []
for i, source in enumerate(comptes_rendus, start=1):
    traduction = traduire_v1(source)

    nombres_source = extraire_nombres(source)
    nombres_sortie = extraire_nombres(traduction)
    manquants = [n for n in nombres_source if n not in nombres_sortie]
    ajoutes = [n for n in nombres_sortie if n not in nombres_source]
    acronymes_perdus = [a for a in ACRONYMES_PROTEGES if a in source and a not in traduction]

    resultats_v1.append({
        "compte_rendu": i,
        "nombres_source": len(nombres_source),
        "nombres_manquants": manquants,
        "nombres_inventes": ajoutes,
        "acronymes_perdus": acronymes_perdus,
        "conforme": not (manquants or ajoutes or acronymes_perdus),
    })

    print(f"--- Compte rendu {i} ---")
    print(traduction)
    print(f"   nombres source : {nombres_source}")
    print(f"   manquants : {manquants} | inventes : {ajoutes} | acronymes perdus : {acronymes_perdus}")
    print()

import pandas as pd
df_v1 = pd.DataFrame(resultats_v1)
print(df_v1[["compte_rendu", "nombres_source", "nombres_manquants",
             "nombres_inventes", "acronymes_perdus", "conforme"]].to_string(index=False))

### V2 — Changement de registre : est-ce que le modele en rajoute ?

Reformuler un compte rendu technique en explication comprehensible par un proche est une demande
frequente. Le piege est connu : en "expliquant", le modele complete. Il ajoute un diagnostic que le
compte rendu ne pose pas, une conduite a tenir, un pronostic rassurant.

Le prompt interdit explicitement ces ajouts, et le controle cherche dans la sortie un vocabulaire
qui ne peut venir que d'un ajout : noms de maladies, traitements, pronostic. Ce controle est
volontairement grossier — il detecte une categorie d'ajout, pas tous. C'est une limite du test,
pas une garantie.

In [ ]:
MOTS_INTERDITS = [
    "epilepsy", "epileptic", "seizure disorder", "diagnosis", "diagnose",
    "treatment", "medication", "drug", "dose", "prognosis", "recovery",
    "should see", "recommend", "cure", "risk of",
]


def vulgariser_v2(texte):
    prompt = f"""Rewrite the following French clinical EEG report as a plain-language explanation
    in French, for a family member with no medical background.

    Strict constraints:
    - Explain ONLY what the report says. Add no diagnosis, no disease name, no treatment,
      no prognosis, no advice, no reassurance.
    - If the report does not state a conclusion, do not produce one.
    - Keep the numbers and durations, but explain what they mean in plain words.
    - Maximum 120 words. Neutral, factual tone.
    - End with exactly this sentence: "Ce texte n'est pas un avis medical."

    Report: ```{texte}```"""
    return get_completion(prompt)


resultats_v2 = []
for i, source in enumerate(comptes_rendus, start=1):
    sortie = vulgariser_v2(source)
    bas = sortie.lower()

    ajouts = [m for m in MOTS_INTERDITS if m in bas]
    phrase_finale = sortie.strip().endswith("Ce texte n'est pas un avis medical.")
    nb_mots = len(sortie.split())

    resultats_v2.append({
        "compte_rendu": i,
        "mots": nb_mots,
        "limite_120_respectee": nb_mots <= 120,
        "phrase_finale_exacte": phrase_finale,
        "ajouts_detectes": ajouts,
        "conforme": not ajouts and phrase_finale and nb_mots <= 120,
    })

    print(f"--- Compte rendu {i} ({nb_mots} mots) ---")
    print(sortie)
    print(f"   ajouts detectes : {ajouts} | phrase finale exacte : {phrase_finale}")
    print()

df_v2 = pd.DataFrame(resultats_v2)
print(df_v2.to_string(index=False))

### V3 — Texte libre vers JSON : le test le plus severe

Extraire une structure d'un texte libre est la transformation la plus utile en pratique — et celle
ou l'hallucination est la plus facile a mesurer, parce qu'un schema attendu existe.

Le corpus de test contient volontairement un cas **incomplet** : le troisieme compte rendu ne
decrit aucun evenement clinique, donc `duree_moyenne_s` et `lateralisation` n'ont pas de valeur. Un
modele honnete met `null`. Un modele qui "complete" invente une valeur plausible — et c'est
precisement l'erreur la plus dangereuse, parce qu'elle est indetectable dans la sortie elle-meme.

Le controle verifie trois choses : le JSON est-il parsable, les champs attendus sont-ils tous
presents, et les champs sans information sont-ils bien a `null`.

In [ ]:
import json as _json

CHAMPS_ATTENDUS = [
    "type_examen", "duree", "nb_canaux", "frequence_rythme_de_fond_hz",
    "anomalies_paroxystiques", "lateralisation", "nb_evenements_cliniques",
    "duree_moyenne_evenement_s",
]


def extraire_json_v3(texte):
    prompt = f"""Extract structured data from the following French clinical EEG report.

    Output a single JSON object, and nothing else - no markdown fences, no commentary.
    Use exactly these keys: {CHAMPS_ATTENDUS}

    Strict rules:
    - Use null for any field the report does not state. Never guess, never infer,
      never fill a field with a plausible value.
    - Copy numbers exactly as written in the report.
    - "anomalies_paroxystiques" is a boolean, or null if the report does not say.
    - "lateralisation" is one of "gauche", "droite", "bilaterale", or null.

    Report: ```{texte}```"""
    return get_completion(prompt)


def analyser_json(brut):
    """Parse la sortie, meme entouree de texte ou de fences markdown."""
    try:
        nettoye = brut.strip().strip("`")
        nettoye = nettoye[nettoye.index("{"): nettoye.rindex("}") + 1]
        return _json.loads(nettoye), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"


resultats_v3 = []
for i, source in enumerate(comptes_rendus, start=1):
    brut = extraire_json_v3(source)
    donnees, erreur = analyser_json(brut)

    print(f"--- Compte rendu {i} ---")
    print(brut)

    if donnees is None:
        print("   JSON illisible :", erreur, "\n")
        resultats_v3.append({"compte_rendu": i, "json_valide": False, "erreur": erreur,
                             "champs_manquants": None, "champs_remplis": None})
        continue

    manquants = [c for c in CHAMPS_ATTENDUS if c not in donnees]
    non_nuls = [c for c, v in donnees.items() if v not in (None, "", [])]

    print(f"   champs manquants : {manquants}")
    print(f"   champs renseignes : {len(non_nuls)}/{len(CHAMPS_ATTENDUS)}\n")

    resultats_v3.append({
        "compte_rendu": i,
        "json_valide": True,
        "erreur": None,
        "champs_manquants": manquants,
        "champs_remplis": len(non_nuls),
    })

df_v3 = pd.DataFrame(resultats_v3)
print(df_v3.to_string(index=False))

#### Le test decisif : le compte rendu 3, celui qui ne dit rien

Le troisieme compte rendu decrit un trace normal, sans evenement clinique. Deux champs n'ont donc
aucune valeur possible dans le texte. On regarde specifiquement ce que le modele en a fait.

In [ ]:
brut3 = extraire_json_v3(comptes_rendus[2])
donnees3, erreur3 = analyser_json(brut3)

print("Sortie brute :\n", brut3, "\n")

if donnees3:
    champs_sans_source = ["nb_evenements_cliniques", "duree_moyenne_evenement_s", "lateralisation"]
    print("Champs qui ne peuvent PAS etre deduits du compte rendu 3 :\n")
    for champ in champs_sans_source:
        valeur = donnees3.get(champ, "<absent>")
        verdict = "OK (null)" if valeur is None else f"VALEUR INVENTEE -> {valeur!r}"
        print(f"   {champ:32} = {valeur!r:20} {verdict}")

    # nb_evenements_cliniques = 0 est defendable (le rapport dit "aucune anomalie"),
    # mais une duree moyenne d'evenement ne peut pas exister s'il n'y a pas d'evenement.
    incoherence = (
        donnees3.get("nb_evenements_cliniques") in (0, None)
        and donnees3.get("duree_moyenne_evenement_s") not in (None, 0)
    )
    print("\nIncoherence interne (duree d'evenement sans evenement) :", incoherence)

### Recapitulatif des trois versions

In [ ]:
recap = pd.DataFrame([
    {"version": "V1 - traduction",     "cas": len(df_v1), "conformes": int(df_v1["conforme"].sum())},
    {"version": "V2 - vulgarisation",  "cas": len(df_v2), "conformes": int(df_v2["conforme"].sum())},
    {"version": "V3 - JSON",           "cas": len(df_v3), "conformes": int(df_v3["json_valide"].sum())},
])
recap["taux"] = (recap["conformes"] / recap["cas"] * 100).round(0).astype(int).astype(str) + " %"
print(recap.to_string(index=False))

---

# Rapport — LLMs for text transformation

## Ce que j'ai teste et pourquoi

Les exemples du cours evaluent les sorties a l'oeil : on lit la traduction, elle a l'air bonne, on
passe a la suite. Sur du texte clinique, ce mode d'evaluation ne tient pas — un chiffre modifie ne
se voit pas dans un texte fluide. J'ai donc construit les trois versions autour d'une meme idee :
**une transformation de texte reussie ne cree rien de nouveau**, et cette propriete peut se
verifier automatiquement.

- **V1 (traduction FR -> EN)** : le controle compare l'ensemble des nombres du texte source et de la
  traduction, et verifie que les acronymes normalises n'ont pas ete traduits. Un nombre manquant,
  ajoute ou modifie fait echouer le cas, meme si la traduction est par ailleurs excellente.
- **V2 (technique -> grand public)** : le controle cherche dans la sortie un vocabulaire qui ne peut
  provenir que d'un ajout — nom de maladie, traitement, pronostic, conseil — et verifie deux
  contraintes de forme (limite de mots, phrase finale imposee au mot pres).
- **V3 (texte libre -> JSON)** : le controle verifie que le JSON est parsable, que tous les champs
  attendus sont la, et surtout ce que le modele fait du troisieme compte rendu, ou deux champs
  n'ont aucune valeur possible dans le texte.

## Resultats

*(A completer apres execution — le tableau recapitulatif et les trois `df_v*` donnent les chiffres.)*

| Version | Cas conformes | Ce qui a echoue |
|---|---|---|
| V1 | ... / 3 | ... |
| V2 | ... / 3 | ... |
| V3 | ... / 3 | ... |

## Ce que la mesure apporte par rapport a une lecture

Trois points ressortent de la conception meme de ces tests, independamment des chiffres obtenus.

**Une contrainte formulee n'est pas une contrainte appliquee.** La phrase finale imposee au mot pres
en V2 est le test le plus revelateur du notebook : si le modele la reformule legerement, c'est que
meme une consigne litterale et triviale n'est qu'une forte suggestion. Tout ce qui repose sur "j'ai
demande au modele de ne pas faire X" herite de cette fragilite.

**L'hallucination la plus dangereuse est celle qui est plausible.** En V3, un `duree_moyenne_s: 45`
sur un compte rendu qui ne decrit aucun evenement est indetectable dans la sortie : la valeur est
bien formee, du bon type, dans un ordre de grandeur credible. Seule une verification externe — ici
la coherence entre deux champs — peut la reperer. C'est pour ca que j'ai ajoute le test
d'incoherence interne plutot que de me contenter de lire le JSON.

**Le format structure ne rend pas le contenu fiable.** Un JSON valide donne une impression de
rigueur qui ne porte que sur la forme. La validation du schema et la validation du contenu sont deux
choses distinctes, et seule la premiere est automatique par nature.

## Limites de mon protocole

Le controle de V2 est le plus faible : une liste de mots interdits detecte une categorie d'ajouts,
pas tous. Un modele peut ajouter une interpretation sans employer aucun des mots de la liste. Un
controle serieux demanderait une comparaison semantique entre source et sortie, ou une evaluation
par un second modele — avec le probleme de fond que cela deplace la question sans la resoudre.

Le corpus est aussi tres petit : trois comptes rendus, une execution. A temperature 0 les sorties
sont stables, mais un taux de conformite sur trois cas n'est pas une mesure, c'est une indication.

## Ce que j'en retiens

Le prompt determine ce que le modele **essaie** de faire ; il ne determine pas ce qu'il fait. Sur
des transformations ou l'exactitude compte, la seule chose sur laquelle on peut s'appuyer est la
verification cote code — et cette verification doit etre concue en meme temps que le prompt, pas
apres. Formuler la contrainte et la mesurer sont deux travaux distincts, et c'est le second qui
manque le plus souvent.

> **Note d'execution** : ce notebook n'a pas ete execute faute de cle OpenAI disponible au moment
> de la redaction. Les cellules sont pretes ; le tableau de resultats et les commentaires factuels
> sont a completer apres un `Run all`.